In [5]:
import csv
import json
import requests
import re
from pathlib import Path
import pandas as pd
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Tuple
import logging

In [6]:
model_name = "deepseek-r1:70b" # "llama3.1:70b"
ollama_url = "http://localhost:11434/api/generate"

input_files = ["./eval_prompts/writingaid_prompts_210825_templ-1.csv.csv",
           "./eval_prompts/writingaid_prompts_210825_templ-2.csv.csv",]

In [7]:
MAX_WORKERS = 12
REQUEST_TIMEOUT = 150
RETRY_ATTEMPTS = 1
TEST_SUBSET = None # Set to None to process all files, or specify a number for a subset

logging.basicConfig(level=logging.INFO, format = '%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
class OllamaProcessor:
    """Optimized Ollama API processor with <think> stripping and yes/no normalization."""
    
    def __init__(self, url: str, model: str, max_workers: int = 8):
        self.url = url
        self.model = model
        self.max_workers = max_workers
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=max_workers,
            pool_maxsize=max_workers * 2,
            max_retries=0
        )
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)
        
    def create_payload(self, prompt: str) -> Dict[str, Any]:
        return {
            "model": self.model,
            "prompt": prompt.strip(),
            "stream": False,
            "options": {
                "temperature": 0.0,
                "top_p": 0.1,
                "num_predict": -1,
                "num_ctx": 4096,
                "repeat_penalty": 1.0
            }
        }
    
    def normalize_response(self, text: str) -> str:
        if not text:
            return "Error: Empty response"
        text = text.strip()
        # Remove any <think>...</think> block
        m = re.search(r'<think>.*?</think>\s*(.*)', text, re.IGNORECASE | re.DOTALL)
        if m:
            text = m.group(1).strip()

        low = text.lower().rstrip('.!?')
        if low.endswith("yes"):
            return "Yes"
        if low.endswith("no"):
            return "No"
        return text
    
    def extract_response_text(self, result: Any) -> str:
        if isinstance(result, dict):
            for key in ("response", "output", "content"):
                if key in result and isinstance(result[key], str):
                    return result[key].strip()
            if "choices" in result and isinstance(result["choices"], list):
                msg = result["choices"][0].get("message", {})
                return msg.get("content", "").strip()
            if "message" in result and isinstance(result["message"], dict):
                return result["message"].get("content", "").strip()
            return str(result)
        if isinstance(result, list) and result:
            return str(result[0])
        return str(result)

    def call_ollama_single(self, prompt: str) -> str:
        if not prompt:
            return "Error: Empty prompt"
        payload = self.create_payload(prompt)
        last_err = None
        for attempt in range(RETRY_ATTEMPTS + 1):
            try:
                resp = self.session.post(self.url, json=payload, timeout=REQUEST_TIMEOUT)
                if resp.status_code == 200:
                    try:
                        result = resp.json()
                        text = self.extract_response_text(result)
                        return self.normalize_response(text)
                    except json.JSONDecodeError as e:
                        return f"JSONError: {e}"
                else:
                    last_err = f"HTTPError: {resp.status_code} - {resp.text[:180]}"
            except requests.exceptions.Timeout:
                last_err = "TimeoutError: Request timed out"
            except requests.exceptions.ConnectionError as e:
                last_err = f"ConnectionError: {e}"
            except Exception as e:
                last_err = f"UnexpectedError: {e}"
        return last_err or "Error: All retry attempts failed"
    
    def process_all_parallel(self, data_items: List[Tuple[int, str]]) -> List[Tuple[int, str]]:
        total = len(data_items)
        completed = 0
        results: List[Tuple[int, str]] = []
        start = time.perf_counter()
        
        logger.info(f"Starting processing of {total} items with {self.max_workers} workers")
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_idx = {executor.submit(self.call_ollama_single, prompt): idx
                             for idx, prompt in data_items}
            
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    res = future.result()
                except Exception as e:
                    res = f"FutureError: {e}"
                    logger.error(f"Error in future for row {idx}: {e}")
                results.append((idx, res))
                completed += 1

                if completed % 5 == 0 or completed == total:
                    elapsed = time.perf_counter() - start
                    avg = elapsed / completed
                    remaining = total - completed
                    eta_sec = remaining * avg
                    eta_min = eta_sec / 60
                    logger.info(
                        f"Progress: {completed}/{total} ({100*completed/total:.1f}%) | Avg: {avg:.1f}s | ETA: {eta_min:.1f}min"
                    )

        return results

In [ ]:
# --- File processing ---
def process_file(input_path: str, processor: OllamaProcessor):
    input_path = str(input_path)
    logger.info(f"Processing {input_path}")
    df = pd.read_csv(input_path, encoding='utf-8')
    if df.empty:
        logger.error(f"No data in {input_path}")
        return
    
    if 'eval_prompt' not in df.columns:
        df.rename(columns={df.columns[-1]: 'eval_prompt'}, inplace=True)
    
    if TEST_SUBSET and len(df) > TEST_SUBSET:
        df = df.head(TEST_SUBSET).copy()
        logger.info(f"TEST MODE: Processing first {len(df)} rows")
    
    df['eval_completion'] = None
    df['model'] = model_name
    
    valid_mask = df['eval_prompt'].notna() & (df['eval_prompt'].astype(str).str.strip() != "")
    valid_indices = df.index[valid_mask].tolist()
    if not valid_indices:
        logger.error("No valid prompts to process.")
        return
    
    data_items = [(idx, str(df.at[idx, 'eval_prompt'])) for idx in valid_indices]
    processor_results = processor.process_all_parallel(data_items)
    
    for idx, completion in processor_results:
        df.at[idx, 'eval_completion'] = completion
    
    in_path = Path(input_path)
    model_tag = model_name.replace(":", "-")
    out_path = in_path.with_name(f"{in_path.stem}_completions_{model_tag}.csv")
    
    df.to_csv(out_path, index=False, encoding='utf-8')
    logger.info(f"Saved to {out_path}")
    return str(out_path)

In [ ]:
# --- Run processing ---
processor = OllamaProcessor(ollama_url, model_name, max_workers=MAX_WORKERS)
outputs = []
for ip in input_files:
    if not Path(ip).exists():
        logger.error(f"File not found: {ip}")
        continue
    out = process_file(ip, processor)
    if out:
        outputs.append(out)

In [ ]:
logger.info(f"Processing complete. Outputs: {outputs}")